- Public score: 0.93828
- Position: 1977 / 2199 ( Top 10% )
- Best Kaggle score: 0.94674

In [ ]:
import os.path

from sklearn.ensemble import RandomForestClassifier
from dotenv import load_dotenv
import pandas as pd
from sklearn.metrics import accuracy_score, classification_report
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
import matplotlib.pyplot as plt
import seaborn as sns
import optuna
from sklearn.model_selection import StratifiedKFold, cross_val_score
import joblib
from sklearn.preprocessing import LabelEncoder
from xgboost import XGBClassifier

In [ ]:
import matplotlib.style as _ms

# Patch for mplcyberpunk expecting matplotlib.style.core
if not hasattr(_ms, "core"):
    _ms.core = _ms

import mplcyberpunk as mpl

load_dotenv()
plt.style.use('cyberpunk')
mpl.add_glow_effects()

In [ ]:
# # Download latest version
#
# path = kagglehub.competition_download('playground-series-s6e9',
#                                       output_dir='data/ev_purchase',
#                                       )
#
# print("Path to competition files:", path)

In [ ]:
test = pd.read_csv('data/ev_purchase/test.csv', index_col='id')
train = pd.read_csv('data/ev_purchase/train.csv', index_col='id')

In [ ]:
train['Total_Station'] = train['Charging_Stations_Near_Home'] + train['Charging_Stations_Near_Work']
train['Charge_Available'] = train['Total_Station'] > 0

train['City_Type'] = train['City_Type'].map({'Urban': 2, 'Suburban': 1, 'Rural': 0})
train['Range_Anxiety_Level'] = train['Range_Anxiety_Level'].map({'High': 2, 'Medium': 1, 'Low': 0})
train['Is_First_Car'] = train['Number_of_Cars_Owned'] == 1

In [ ]:
train['Total_Station'] = train['Charging_Stations_Near_Home'] + train['Charging_Stations_Near_Work']
train['Charge_Available'] = train['Total_Station'] > 0

train['City_Type'] = train['City_Type'].map({'Urban': 2, 'Suburban': 1, 'Rural': 0})
train['Range_Anxiety_Level'] = train['Range_Anxiety_Level'].map({'High': 2, 'Medium': 1, 'Low': 0})
train['Is_First_Car'] = train['Number_of_Cars_Owned'] == 1

### Feature Engineering

In [ ]:
train.describe()

In [ ]:
train.head(10)

In [ ]:
cat_columns = train.select_dtypes(include=str).columns
# we need check unique values in categorical columns to see if there are any inconsistencies

train[cat_columns].nunique()

In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), ['Age',
                                   'Annual_Income_USD',
                                   'Daily_Commute_km',
                                   'Number_of_Cars_Owned',
                                   'Charging_Stations_Near_Home',
                                   'Charging_Stations_Near_Work',
                                   'Environmental_Concern_Level'
                                   ]),
        ('label', OrdinalEncoder(), ['Gender',
                                     'Home_Charging_Possible',
                                     'Subsidy_Available',
                                     'Range_Anxiety_Level']),
        ('cat', OneHotEncoder(handle_unknown='ignore'), ['City_Type',
                                                         'Current_Car_Type']),
    ]
)

In [ ]:
pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(n_estimators=100,
                                          random_state=42,
                                          n_jobs=-1))
])

In [ ]:
X_train, X_val, y_train, y_val = train_test_split(
    train.drop(columns=['Will_Buy_EV']),
    train['Will_Buy_EV'],
    test_size=0.2,
    random_state=42
)

In [ ]:
# save trained pipeline to disk

model_name = 'ev_purchase_pipeline.pkl'

if os.path.exists(model_name):
    print(f'Model file {model_name} already exists. Loading the existing model.')
    pipeline = joblib.load(model_name)
else:
    print(f'Model file {model_name} does not exist. Creating a new model.')
    pipeline.fit(X_train, y_train)
    joblib.dump(pipeline, model_name)
    print("Pipeline saved to 'ev_purchase_pipeline.pkl'")

In [ ]:
y_pred_pipeline = pipeline.predict(X_val)
accuracy_pipeline = accuracy_score(y_val, y_pred_pipeline)
print(f"Validation Accuracy (Pipeline): {accuracy_pipeline:.4f}")

In [ ]:
conf_matrix = pd.crosstab(y_val, y_pred_pipeline, rownames=['Actual'], colnames=['Predicted'])
sns.heatmap(conf_matrix, annot=True, fmt='d', cmap='Blues')
plt.title(f'Confusion Matrix (Validation Set) - Accuracy: {accuracy_pipeline:.4f}')
plt.show()

In [ ]:
clf_report = classification_report(y_val, y_pred_pipeline)
print(clf_report)

In [ ]:
# Feature importance from the Random Forest model
feature_importances = pipeline.named_steps['classifier'].feature_importances_
# Get feature names from the preprocessor
num_features = ['Age', 'Annual_Income_USD', 'Daily_Commute_km', 'Number_of_Cars_Owned', 'Charging_Stations_Near_Home', 'Charging_Stations_Near_Work', 'Environmental_Concern_Level']
label_features = ['Gender', 'Home_Charging_Possible', 'Subsidy_Available', 'Range_Anxiety_Level']
cat_features = pipeline.named_steps['preprocessor'].named_transformers_['cat'].get_feature_names_out(['City_Type', 'Current_Car_Type']).tolist()
all_features = num_features + label_features + cat_features
# Create a DataFrame for feature importances
feature_importance_df = pd.DataFrame({
    'Feature': all_features,
    'Importance': feature_importances
}).sort_values(by='Importance', ascending=False)

In [ ]:
# Plot feature importances
plt.figure(figsize=(12, 8))
sns.barplot(x='Importance', y='Feature', data=feature_importance_df)
plt.title('Feature Importances from Random Forest Classifier')
plt.show()

In [ ]:
feature_importance_df

In [ ]:
# let's prepare the data
# 1. use small piece of the training data ( 1000 rows ) to speed up the hyperparameter optimization process
X_train_small, _, y_train_small, _ = train_test_split(
    X_train,
    y_train,
    train_size=1000,
    random_state=42
)

# 2. Transform the small training data using the preprocessor
X_train_small_transformed = preprocessor.fit_transform(X_train_small)

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

In [ ]:
# use Optuna to optimize hyperparameters of the Random Forest model
def objective(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 100, 500),
        'max_depth': trial.suggest_int('max_depth', 10, 30),
        'min_samples_split': trial.suggest_int('min_samples_split', 5, 20),
        'min_samples_leaf': trial.suggest_int('min_samples_leaf', 1, 10),
        'criterion': trial.suggest_categorical('criterion', ['gini', 'entropy']),
    }

    model = RandomForestClassifier(**params, random_state=42)
    cv_scores = cross_val_score(model,
                                X_train_small_transformed,
                                y_train_small,
                                cv=cv,
                                scoring='accuracy')
    accuracy = cv_scores.mean()

    if trial.should_prune():
        raise optuna.exceptions.TrialPruned()

    return accuracy

In [ ]:
pruner = optuna.pruners.MedianPruner()
study = optuna.create_study(direction='maximize',
                            study_name='RandomForest_Optimization',
                            sampler=optuna.samplers.TPESampler(seed=42),
                            pruner=pruner)

study.optimize(objective,
               n_trials=50,
               n_jobs=-1,
               show_progress_bar=True)

print("Best hyperparameters: ", study.best_params)

In [ ]:
# Visualize the optimization history
optuna.visualization.plot_optimization_history(study)

In [ ]:
optuna.visualization.plot_param_importances(study)

In [ ]:
optuna.visualization.plot_parallel_coordinate(study)

In [ ]:
score = study.best_value
print(f'Best cross-validated accuracy: {score:.4f}')

In [ ]:
# Train the final model with the best hyperparameters
best_params = study.best_params
# best_params =  {'n_estimators': 425, 'max_depth': 22, 'min_samples_split': 7, 'min_samples_leaf': 4, 'criterion': 'entropy'}

final_clf = RandomForestClassifier(
    **best_params,
    random_state=42,
    n_jobs=-1
)

final_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', final_clf)
])

final_pipeline.fit(X_train, y_train)


In [ ]:
y_pred_final = final_pipeline.predict(X_val)
final_accuracy = accuracy_score(y_val, y_pred_final)
print(f'Final Model Validation Accuracy: {final_accuracy:.4f}')

In [ ]:
y_pred_final = final_pipeline.predict(X_train)
final_accuracy = accuracy_score(y_train, y_pred_final)
print(f'Final Model Training Accuracy: {final_accuracy:.4f}')

In [ ]:
# Let's use full training data to train the final model and make predictions on the test set
X, y = train.drop(columns=['Will_Buy_EV']), train['Will_Buy_EV']
final_pipeline.fit(X ,y)

In [ ]:
y_pred_sub = final_pipeline.predict_proba(test)[:, 1]
submission = pd.DataFrame({'id': test.index, 'Will_Buy_EV': y_pred_sub})
submission.to_csv('submission.csv', index=False)

In [ ]:



le = LabelEncoder()
y_train_small = le.fit_transform(y_train_small)
y_val = le.transform(y_val)

In [ ]:
# Train an XGBoost model for comparison
xgb_model = XGBClassifier(
    n_estimators=100,
    max_depth=6,
    learning_rate=0.1,
    random_state=42,
    eval_metric='logloss'
)

xgb_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', xgb_model)
])
xgb_pipeline.fit(X_train, y_train)

In [ ]:
y_pred_xgb = xgb_pipeline.predict(X_val)
xgb_accuracy = accuracy_score(y_val, y_pred_xgb)
print(f"Validation Accuracy (XGBoost): {xgb_accuracy:.4f}")

In [ ]:
# Feature importance from the XGBoost model
xgb_feature_importances = xgb_pipeline.named_steps['classifier'].feature_importances_
# Create a DataFrame for feature importances
xgb_feature_importance_df = pd.DataFrame({
    'Feature': all_features,
    'Importance': xgb_feature_importances
}).sort_values(by='Importance', ascending=False)

In [ ]:
# Plot feature importances for XGBoost
plt.figure(figsize=(12, 8))
sns.barplot(x='Importance', y='Feature', data=xgb_feature_importance_df)
plt.title('Feature Importances from XGBoost Classifier')
plt.show()

In [ ]:
# # Tune hyperparameters for XGBoost using Optuna
# def xgb_objective(trial):
#     params = {
#         "n_estimators": trial.suggest_int("n_estimators", 100, 1000),
#         "max_depth": trial.suggest_int("max_depth", 3, 12),
#         "learning_rate": trial.suggest_float("learning_rate", 0.005, 0.3, log=True),
#         "subsample": trial.suggest_float("subsample", 0.6, 1.0),
#         "colsample_bytree": trial.suggest_float("colsample_bytree", 0.3, 1.0),
#         "gamma": trial.suggest_float("gamma", 0.0, 10.0),
#         "min_child_weight": trial.suggest_int("min_child_weight", 1, 30),
#         "reg_alpha": trial.suggest_float("reg_alpha", 1e-8, 10.0, log=True),
#         "reg_lambda": trial.suggest_float("reg_lambda", 1e-3, 20.0, log=True),
#         "max_delta_step": trial.suggest_int("max_delta_step", 0, 20),
#         # "tree_method": "hist",
#         # "eval_metric": "logloss",
#     }
#
#     model = XGBClassifier(
#         **params,
#         random_state=42,
#         eval_metric='logloss'
#     )
#     # Train  the model on pre-transformed data limited data
#     cv_scores = cross_val_score(model,
#                                 X_train_small_transformed,
#                                 y_train_small,
#                                 cv=cv,
#                                 scoring='accuracy')
#     accuracy = cv_scores.mean()
#
#     if trial.should_prune():
#         raise optuna.exceptions.TrialPruned()
#
#     return accuracy

In [ ]:
def xgb_objective(trial):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 100, 1000),
        "max_depth": trial.suggest_int("max_depth", 3, 12),
        "learning_rate": trial.suggest_float("learning_rate", 0.005, 0.3, log=True),
        "subsample": trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.3, 1.0),
        "min_child_weight": trial.suggest_int("min_child_weight", 1, 30),
        "gamma": trial.suggest_float("gamma", 0.0, 30.0),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-8, 30.0, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-3, 30.0, log=True),
        # "eval_metric": "logloss",
        # "tree_method": "hist",
        "random_state": 42,
    }

    model = XGBClassifier(**params)

    scores = []
    for fold, (tr_idx, va_idx) in enumerate(cv.split(X_train_small_transformed, y_train_small)):
        X_tr, X_va = X_train_small_transformed[tr_idx], X_train_small_transformed[va_idx]
        y_tr, y_va = y_train_small[tr_idx], y_train_small[va_idx]

        model.fit(
            X_tr, y_tr,
            eval_set=[(X_va, y_va)],
            verbose=False,
            # early_stopping_rounds=20,
        )

        score = model.score(X_va, y_va)
        scores.append(score)
        trial.report(sum(scores) / len(scores), step=fold)

        if trial.should_prune():
            raise optuna.exceptions.TrialPruned()

    return sum(scores) / len(scores)

In [ ]:
pruner = optuna.pruners.MedianPruner()

xgb_study = optuna.create_study(direction='maximize',
                                 study_name='XGBoost_Optimization',
                                 sampler=optuna.samplers.TPESampler(seed=42),
                                 pruner=pruner)

xgb_study.optimize(xgb_objective,
                   n_trials=150,
                   n_jobs=-1,
                   show_progress_bar=True)

In [ ]:
best_score = xgb_study.best_value
best_params = xgb_study.best_params
print(f'Best cross-validated accuracy for XGBoost: {best_score:.4f}')
print("Best hyperparameters for XGBoost: ", best_params)

In [ ]:
# Visualize the optimization history for XGBoost
optuna.visualization.plot_optimization_history(xgb_study)

In [ ]:
# Visualize parameter importances for XGBoost
optuna.visualization.plot_param_importances(xgb_study)

In [ ]:
# Visualize parallel coordinate plot for XGBoost
optuna.visualization.plot_parallel_coordinate(xgb_study)

In [ ]:
# Best hyperparameters for XGBoost:  {'n_estimators': 309, 'max_depth': 9, 'learning_rate': 0.2613765708422029, 'subsample': 0.9802642633747923, 'colsample_bytree': 0.5827220970702967, 'gamma': 2.264710867164483}